# Commons Simulation - ANOVA Analysis

Bu notebook `output/model_metrics_104x25.csv` dosyasindan ANOVA analizi uretir.

In [ ]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

In [ ]:
model_path = 'output/model_metrics_104x25.csv'
df = pd.read_csv(model_path)
print(df.shape)
df.head()

In [ ]:
# Her run icin son tik metriklerini al
final_df = (
    df.sort_values(['scenario_id', 'repeat_id', 'tick'])
    .groupby(['scenario_id', 'repeat_id'], as_index=False)
    .tail(1)
    .copy()
)
print(final_df.shape)
final_df[['system_type', 'config_name', 'gini_coefficient', 'mean_trust', 'resource_utilization']].head()

In [ ]:
# ANOVA: sistem tipinin Gini uzerindeki etkisi
model_gini = ols(
    'gini_coefficient ~ C(system_type) + C(cohesion) + N_people + N_resources',
    data=final_df,
).fit()
anova_gini = sm.stats.anova_lm(model_gini, typ=2)
anova_gini

In [ ]:
# Eta-squared
anova_gini = anova_gini.copy()
ss_total = anova_gini['sum_sq'].sum()
anova_gini['eta_sq'] = anova_gini['sum_sq'] / ss_total
anova_gini

In [ ]:
# Tukey HSD - sistem tipleri arasi Gini farklari
tukey = pairwise_tukeyhsd(
    endog=final_df['gini_coefficient'],
    groups=final_df['system_type'],
    alpha=0.05,
)
print(tukey)

In [ ]:
# Benzer analiz mean_trust icin
model_trust = ols(
    'mean_trust ~ C(system_type) + C(cohesion) + N_people + N_resources',
    data=final_df,
).fit()
sm.stats.anova_lm(model_trust, typ=2)